# Quran-ARS — Free GPU Test (Google Colab)

Test the recitation-grading service on a **free GPU** before you buy a server.

**Setup (do this first):** top menu → *Runtime* → *Change runtime type* → set **Hardware
accelerator: GPU** (a free **T4** works; L4/A100 are faster). Then *Runtime* → **Run all**.

What this notebook does, automatically:
1. installs the service + the NAMAA model's deps,
2. loads the model and starts the API,
3. exposes a **public URL** (via cloudflared — no signup),
4. runs **two tests** — the simple sync endpoint *and* the real async `/api/evaluate` + webhook,
   printing the score, the learner's **diacritized** recitation, and any **harakat/tajweed** errors.

Notes: the session is temporary (~12 h, disconnects on idle). On a **T4**, bf16 runs but is
*slow* (~10–30 s/request) — that's fine for testing; an L4/A100 is much faster. `AI_API_KEY`
below is a throwaway test value.

In [ ]:
# 1) Confirm you have a GPU (if this errors: Runtime > Change runtime type > GPU)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2) Get the code + install the service dependencies (~2-3 min)
import os
if not os.path.exists('Quran-ARS'):
    !git clone -q https://github.com/HassanAbdelshafy21/Quran-ARS.git
# The 2B model needs transformers>=5.4. Colab already has torch+CUDA, so we install everything
# EXCEPT torch (to avoid a slow reinstall).
!pip install -q -U "transformers>=5.4" accelerate sentencepiece protobuf safetensors \
    fastapi "uvicorn[standard]" python-multipart httpx edge_tts librosa soundfile jiwer moviepy requests
# cloudflared: a free public tunnel (no account needed)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
print("\n✓ dependencies installed")

In [ ]:
# 3) Start a public tunnel + the API server, then wait for the model to load (~1-3 min on T4)
import subprocess, time, re, os, requests

AI_API_KEY = "test-key-123"          # throwaway secret for this test
DELIV = "Quran-ARS/delivery"

# 3a) public URL via cloudflared (output -> file, so the pipe never blocks)
ctun = open("cloudflared.log", "w+")
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                 stdout=ctun, stderr=ctun, text=True)
public_url = None
for _ in range(40):
    ctun.seek(0); m = re.search(r"https://[-\w]+\.trycloudflare\.com", ctun.read())
    if m: public_url = m.group(0); break
    time.sleep(2)
print("public URL:", public_url or "(tunnel not ready — the local test still works)")

# 3b) start the FastAPI server (loads NAMAA). Logs -> server.log
env = dict(os.environ, AI_API_KEY=AI_API_KEY,
           PUBLIC_BASE_URL=public_url or "http://localhost:8000", HF_HUB_OFFLINE="0")
logf = open("server.log", "w")
subprocess.Popen(["python", "main.py"], cwd=DELIV, env=env, stdout=logf, stderr=logf, text=True)

# 3c) wait for /health
ok = False
for i in range(60):
    try:
        h = requests.get("http://localhost:8000/health", timeout=5).json()
        if h.get("model_loaded"): print("\n✓ HEALTHY:", h); ok = True; break
    except Exception:
        pass
    time.sleep(5)
if not ok:
    print("\nstill loading — last server log lines:")
    !tail -n 15 server.log

In [ ]:
# 4) SYNC test — grade one sample recitation (Surah 112:1, a correct Husary recitation)
import requests
open("sample.mp3", "wb").write(requests.get("https://everyayah.com/data/Husary_128kbps/112001.mp3").content)
r = requests.post("http://localhost:8000/grade_recitation",
                  files={"file": open("sample.mp3", "rb")},
                  data={"target_ayah": "قُلْ هُوَ ٱللَّهُ أَحَدٌ", "surah_num": 112, "ayah_num": 1},
                  timeout=600).json()
print("score      :", round(r["accuracy"]*100), "%   passed:", r["passed"])
print("recitation :", r["user_recitation_diacritized"])
print("harakat    : checked", r["harakat_checked"], "word(s),", len(r["harakat_errors"]), "flagged")
for e in r["harakat_errors"]:
    d = e["details"][0]; print("   -", e["word"], ": letter", d["letter"], "said", d["got"], "expected", d["expected"])

In [ ]:
# 5) ASYNC test — the REAL integration: POST /api/evaluate, then receive the webhook callback
import threading, json, time, requests
from http.server import BaseHTTPRequestHandler, HTTPServer

got = {}
class _H(BaseHTTPRequestHandler):
    def do_POST(self):
        n = int(self.headers.get("Content-Length", 0)); got["payload"] = json.loads(self.rfile.read(n))
        self.send_response(200); self.end_headers(); self.wfile.write(b"{}")
    def log_message(self, *a): pass
threading.Thread(target=HTTPServer(("0.0.0.0", 9099), _H).serve_forever, daemon=True).start()

body = {"audioUrl": "https://everyayah.com/data/Husary_128kbps/112001.mp3",
        "surahNumber": 112, "surahName": "الإخلاص", "fromAyah": 1, "toAyah": 1,
        "userId": 1, "recitationId": 1,
        "webhookUrl": "http://localhost:9099/webhook", "webhookSecret": "test-secret"}
resp = requests.post("http://localhost:8000/api/evaluate", json=body,
                     headers={"Authorization": f"Bearer {AI_API_KEY}"}, timeout=30)
print("immediate response:", resp.status_code, resp.json())

print("\nwaiting for the webhook callback ...")
for _ in range(90):
    if "payload" in got: break
    time.sleep(2)
print(json.dumps(got.get("payload", {"_": "no webhook received — see server.log"}), ensure_ascii=False, indent=2))

## What you just saw / what to check
- **`/health` → `model_loaded: true`** means the 2B model is running on the free GPU.
- **SYNC** returns a score + the learner's **diacritized** recitation + harakat feedback.
- **ASYNC** is the exact backend contract: `/api/evaluate` returns instantly (`status: processing`,
  a `jobId`), and the full result arrives at your `webhookUrl` under `data` (with
  `userRecitationDiacritized` and `harakatErrors`).

### Use the public URL with your app
The **public URL** printed in step 3 (`https://…trycloudflare.com`) is a live endpoint while this
notebook runs — your backend can call `POST <public_url>/api/evaluate` with
`Authorization: Bearer test-key-123` to try the real integration end-to-end, for free.

### When you're ready for production
This proves the model + API work. For the *exact* production path (the Docker container), do one
run on a **Google Cloud $300 free trial** VM with `docker compose up -d --build` — then buy the
box. Everything the backend needs is in `Quran-ARS/delivery/` (see `AGENT_DEPLOY_PROMPT.md`).

### Troubleshooting
- Model slow to load / test times out on **T4** → normal (bf16 isn't accelerated on T4); wait, or
  pick a faster GPU. `!tail -n 40 server.log` shows what the server is doing.
- `401` on `/api/evaluate` → the `Authorization` header must be `Bearer test-key-123` (the
  `AI_API_KEY` set in step 3).